# NASA Intelligence Chat — Live Demo

This notebook demonstrates the full RAG pipeline end-to-end:
1. Connect to the ChromaDB collection
2. Retrieve relevant chunks for a question
3. Generate a grounded, cited answer
4. (Optional) Evaluate with RAGAS

> **Prerequisites:** Run `python scripts/build_index.py --update-mode replace` once to populate ChromaDB,  
> and set `OPENAI_API_KEY` in your `.env` file.

---

## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from nasa_rag.config import get_settings
cfg = get_settings()

if not cfg.is_configured():
    print('⚠️  OPENAI_API_KEY not set — LLM generation will be skipped.')
else:
    print('✅ API key loaded.')
print(f'   Model: {cfg.chat_model}')
print(f'   ChromaDB: {cfg.chroma_dir}')

✅ API key loaded.
   Model: gpt-4o-mini
   ChromaDB: ./chroma_db


## 2. Connect to ChromaDB

In [2]:
from nasa_rag.retrieval import discover_chroma_backends, initialize_collection

backends = discover_chroma_backends(PROJECT_ROOT)

if not backends:
    print('❌ No ChromaDB collections found.')
    print('   Run: python scripts/build_index.py --update-mode replace')
else:
    for key, info in backends.items():
        print(f'  Found: {info["display_name"]}')

  Found: nasa_missions (7898 docs) [chroma_db]


In [3]:
# Connect to the first available collection
if backends:
    backend = list(backends.values())[0]
    collection, ok, err = initialize_collection(
        backend['directory'], backend['collection_name']
    )
    if ok:
        print(f'✅ Connected — {collection.count()} chunks indexed.')
    else:
        print(f'❌ Error: {err}')
        collection = None
else:
    collection = None

✅ Connected — 7898 chunks indexed.


## 3. Semantic retrieval

In [4]:
from nasa_rag.retrieval import retrieve_documents, format_context

QUESTION = 'What caused the Apollo 13 oxygen tank explosion?'

if collection:
    results = retrieve_documents(
        collection,
        query=QUESTION,
        n_results=5,
        mission_filter='apollo 13',
    )

    if results and results.get('documents'):
        docs = results['documents'][0]
        metas = results['metadatas'][0]
        print(f'Retrieved {len(docs)} chunks:\n')
        for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
            print(f'[{i}] {meta.get("mission","").replace("_"," ").title()} — '
                  f'{meta.get("document_category","").replace("_"," ").title()}')
            print(f'    Chunk {meta.get("chunk_index","?")}/{meta.get("total_chunks","?")}')
            print(f'    {doc[:120]}…')
            print()
    else:
        docs, metas = [], []
        print('No documents retrieved.')
else:
    docs, metas = [], []
    print('No collection available.')

Retrieved 5 chunks:

[1] Apollo 13 — Public Affairs Officer
    Chunk 1176/1673
    being considered and discussed. At the present time Apollo 13
is 139 164 nautical miles from the earth travelling at a
s…

[2] Apollo 13 — Public Affairs Officer
    Chunk 554/1673
    ime the LM oxygen quantity is 50.6 pounds and 300 pounds of
water on board. Cabin pressure holding steadily and 4.7 up
t…

[3] Apollo 13 — Public Affairs Officer
    Chunk 1242/1673
    e spacecraft.
PAO
Apparently it was an accidental nudge of
the king switch at the capsule communicator's console causing…

[4] Apollo 13 — Public Affairs Officer
    Chunk 1246/1673
    m
this is still a sealed dual ring system in the Command Module
for attitude control during entry. There are 245 pounds …

[5] Apollo 13 — Public Affairs Officer
    Chunk 553/1673
    of it in order to conserve the
oxygen.
CAPCOM Okay. We just want you to turn off the
water accumulators and not the glyc…



## 4. Format context string

In [5]:
context_str = format_context(docs, metas) if docs else ''
print(context_str[:1500] if context_str else '(empty — no documents retrieved)')

=== RETRIEVED NASA MISSION CONTEXT ===

[Source 1]
Mission: Apollo 13
File: data_text/apollo13/AS13_PAO_textract_full_text.txt
Category: Public Affairs Officer
Chunk: 1176/1673
Content:
being considered and discussed. At the present time Apollo 13
is 139 164 nautical miles from the earth travelling at a
speed of 4745 feet per second. Also like to clarify again
the situation with respect to the super critical helium tank
and the burst disk which ruptured at about 180 hours when
the pressure got up to some 1937 pounds per square inch. This
burst disk is in the tank for the express purpose of keeping

--- PAGE 670 ---
APOLLO 13 MISSION COMMENTARY 4-16-70 CST 3:19A GET 110:06:00 474/2
PAO
pressure from going above levels which
the tank can withstand. Had the disk failed to rupture,
we had a backup procedure worked out whereby the tank would
be vented manually. Of course, it was not necessary to
put this backup into effect because the disk ruptured at
about the level it was expected it woul

## 5. Generate grounded answer

In [6]:
from nasa_rag.generation import generate_response

if cfg.is_configured() and context_str:
    answer = generate_response(
        openai_key=cfg.openai_api_key,
        user_message=QUESTION,
        context=context_str,
        conversation_history=[],
        model=cfg.chat_model,
    )
    print('QUESTION:')
    print(f'  {QUESTION}')
    print()
    print('ANSWER:')
    print(answer)
else:
    answer = ''
    print('Skipped — API key not set or no context available.')

QUESTION:
  What caused the Apollo 13 oxygen tank explosion?

ANSWER:
The available NASA documents do not provide sufficient information to answer this question.


## 6. Multi-question demo

In [7]:
DEMO_QUESTIONS = [
    ('Who were the crew members of Apollo 11?',       None),
    ('What caused the Challenger disaster?',           'challenger'),
    ('How did Mission Control respond to Apollo 13?', 'apollo 13'),
]

if cfg.is_configured() and collection:
    for q, mission_filter in DEMO_QUESTIONS:
        print('=' * 70)
        print(f'Q: {q}')
        r = retrieve_documents(collection, q, n_results=4, mission_filter=mission_filter)
        ctx = ''
        if r and r.get('documents') and r['documents'][0]:
            ctx = format_context(r['documents'][0], r['metadatas'][0])
        a = generate_response(
            cfg.openai_api_key, q, ctx, [], model=cfg.chat_model
        )
        print(f'A: {a[:500]}…' if len(a) > 500 else f'A: {a}')
        print()
else:
    print('Skipped — API key not set or ChromaDB not available.')

Q: Who were the crew members of Apollo 11?
A: The crew members of Apollo 11 were Neil A. Armstrong, who served as the Commander; Michael Collins, the Command Module Pilot; and Edwin E. Aldrin, Jr., the Lunar Module Pilot [Source 1].

Q: What caused the Challenger disaster?
A: The available NASA documents do not provide sufficient information to answer this question.

Q: How did Mission Control respond to Apollo 13?
A: Mission Control responded to Apollo 13 by maintaining communication with the crew and providing updates on the spacecraft's status. For example, at 52 minutes into the flight, Mission Control reacquired the spacecraft's signal and confirmed that everything was proceeding well with the mission timeline. The capsule communicator (CAP COM) acknowledged the crew's reports and provided instructions, indicating that the communication was clear and effective ([Source 2]).

Additionally, during a later …



## 7. RAGAS evaluation (optional)

In [8]:
from nasa_rag.evaluation import evaluate_response, RAGAS_AVAILABLE

if not RAGAS_AVAILABLE:
    print('RAGAS not installed. Run: pip install ragas')
elif not cfg.is_configured() or not answer:
    print('Skipped — API key or answer not available.')
else:
    print('Evaluating response quality with RAGAS (may take 20–40s)…')
    scores = evaluate_response(
        question=QUESTION,
        answer=answer,
        contexts=docs,
        openai_key=cfg.openai_api_key,
    )
    if 'error' in scores:
        print(f'Evaluation error: {scores["error"]}')
    else:
        print('RAGAS Scores:')
        for metric, value in scores.items():
            bar = '█' * int(value * 20)
            print(f'  {metric:<25}: {value:.3f}  [{bar:<20}]')

Evaluating response quality with RAGAS (may take 20–40s)…


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: TimeoutError()
RAGAS evaluation timed out after 180s.


Evaluation error: Evaluation timed out after 180s.
